# Interactive Satellite Image Retrieval & ML Pipeline

Click anywhere on the map to select a location, then retrieve satellite imagery
and run it through the vegetation restoration ML pipeline.

## 0. Dependency checks

In [ ]:
import subprocess, sys, importlib

_REQUIRED = {
    "geemap":       "geemap",
    "geopy":        "geopy",
    "ipyleaflet":   "ipyleaflet",
    "earthengine":  "earthengine-api",
}

_missing = []
for pkg_name, import_name in _REQUIRED.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        _missing.append(pkg_name)

if _missing:
    print(f"Installing missing packages: {', '.join(_missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *_missing])
    print("Done.")
else:
    print("All required packages are installed.")

## Initialize Google Earth Engine (skip if already done)

In [ ]:
import ee

try:
    ee.data._credentials
    print("GEE already initialized.")
except AttributeError:
    ee.Initialize(project="urban-green-mapping")
    print("GEE initialized with project 'urban-green-mapping'.")

## Imports

In [ ]:
import json
import logging
from datetime import datetime

import ipywidgets as widgets
from IPython.display import display, clear_output

import geemap

from utils.interactive_collection import (
    reverse_geocode,
    fetch_satellite_image,
    extract_band_values,
    build_tile,
    run_ml_pipeline,
)
from utils.config_loader import load_config, resolve_config_path

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Stage 1: Interactive Map — Click to Select a Location

In [ ]:
# State variables
selected_lat = None
selected_lng = None
target_date = "2025-10-01"  # default

# Create the interactive map centred on Himachal Pradesh
center = [32.30, 77.20]
m = geemap.Map(center=center, zoom=9)

# Info widget to display coordinates
info = widgets.HTML(value="<b>Click on the map</b> to select a location.")
m.add(info, bottom_left=True)


def on_map_click(**kwargs):
    """Callback when user clicks the map."""
    global selected_lat, selected_lng
    output = kwargs.get("output")
    if output is None:
        return
    # output is [lng, lat]
    lng, lat = output[0], output[1]
    selected_lat = round(lat, 6)
    selected_lng = round(lng, 6)

    # Add/update a marker
    marker = geemap.Markers(position=[lat, lng], color="orange")
    m.add(marker)

    info.value = (
        f"<b>Selected:</b> {selected_lat}, {selected_lng} "
        f"| <i>Press 'Run Pipeline' to continue</i>"
    )
    print(f"Clicked coordinates: lat={selected_lat}, lng={selected_lng}")


m.on_click(on_map_click)
display(m)

### Target Date Input

In [ ]:
date_box = widgets.Text(
    value=target_date,
    description="Target date:",
    tooltip="YYYY-MM-DD",
    layout=widgets.Layout(width="220px"),
)

run_btn = widgets.Button(
    description="Run Pipeline",
    button_style="success",
    layout=widgets.Layout(margin="0 0 0 10px"),
)

status_label = widgets.HTML(value="")

display(widgets.HBox([date_box, run_btn]))
display(status_label)

## Stages 2-5: Pipeline (triggered by button click)

In [ ]:
def set_status(msg: str, ok: bool = True):
    color = "green" if ok else "red"
    status_label.value = f'<span style="color:{color}"><b>{msg}</b></span>'


def run_pipeline(_btn=None):
    global selected_lat, selected_lng, target_date
    target_date = date_box.value.strip()

    # --- Validation ---
    if selected_lat is None or selected_lng is None:
        set_status("Please click on the map first.", ok=False)
        return
    try:
        datetime.strptime(target_date, "%Y-%m-%d")
    except ValueError:
        set_status("Invalid date format. Use YYYY-MM-DD.", ok=False)
        return

    config = load_config(resolve_config_path(None))

    # ===== Stage 2: Reverse Geocoding =====
    set_status("Stage 2: Reverse geocoding...")
    try:
        loc_info = reverse_geocode(selected_lat, selected_lng)
        print("\n--- Reverse Geocoding ---")
        print(f"  Location : {loc_info['location_name']}")
        print(f"  City     : {loc_info['city']}")
        print(f"  Country  : {loc_info['country']}")
        print(f"  Address  : {loc_info['full_address']}")
        set_status("Reverse geocoding complete.")
    except Exception as e:
        set_status(f"Geocoding failed: {e}", ok=False)
        return

    # ===== Stage 3: GEE Image Retrieval =====
    set_status("Stage 3: Fetching satellite image from GEE...")
    try:
        gee_result = fetch_satellite_image(
            selected_lat, selected_lng, target_date
        )
        print(f"\n--- Satellite Image ---")
        print(f"  Source    : {gee_result['source']}")
        print(f"  Image date: {gee_result['image_date']}")
    except Exception as e:
        set_status(f"GEE image fetch failed: {e}", ok=False)
        return

    # Display satellite image layer on the map
    try:
        m.add_layer(
            gee_result["image_display"],
            gee_result["viz_params"],
            gee_result["rgb_bands"],
            f"{gee_result['source']} {gee_result['image_date']}",
        )
        set_status("Satellite image displayed on map.")
    except Exception as e:
        logger.warning("Could not display image layer: %s", e)

    # ===== Stage 4a: Extract band values =====
    set_status("Stage 4: Extracting band values...")
    try:
        band_values = extract_band_values(gee_result)
        print("\n--- Band Values at Clicked Point ---")
        for b, v in band_values.items():
            print(f"  {b}: {v:.2f}")
        set_status("Band values extracted.")
    except Exception as e:
        set_status(f"Band extraction failed: {e}", ok=False)
        return

    # ===== Stage 4b: Build tile for ML pipeline =====
    set_status("Stage 4b: Building tile for ML pipeline...")
    try:
        tile = build_tile(gee_result, config)
        print(f"\n--- Tile Built ---")
        print(f"  tile_id  : {tile['tile_id']}")
        print(f"  sentinel : {tile['sentinel'].shape}")
        print(f"  dem      : {tile['dem'].shape}")
        set_status("Tile built successfully.")
    except Exception as e:
        set_status(f"Tile building failed: {e}", ok=False)
        import traceback
        traceback.print_exc()
        return

    # ===== Stage 4c: Run ML pipeline =====
    set_status("Stage 4c: Running ML inference pipeline...")
    try:
        ml_result = run_ml_pipeline(tile, config)
        print("\n--- ML Pipeline Results ---")
        print(f"  Tile ID           : {ml_result['tile_id']}")
        print(f"  Vegetation score  : {ml_result['vegetation_score']:.4f}")
        print(f"  Avalanche score   : {ml_result['avalanche_score']:.4f}")
        print(f"  Combined score    : {ml_result['combined_score']:.4f}")
        print(f"  Class probabilities: {ml_result['probabilities'].tolist()}")
        set_status("ML pipeline complete.")
    except Exception as e:
        set_status(f"ML pipeline failed: {e}", ok=False)
        import traceback
        traceback.print_exc()
        return

    # ===== Stage 5: Summary Panel =====
    band_str = ", ".join(f"{k}={v:.1f}" for k, v in band_values.items())
    summary_html = (
        f"<div style='padding:10px; border:1px solid #ccc; border-radius:6px; "
        f"background:#f9f9f9; font-size:14px;'>"
        f"<h3 style='margin:0 0 8px 0;'>Results Summary</h3>"
        f"<b>Coordinates:</b> {selected_lat}, {selected_lng}<br>"
        f"<b>Location:</b> {loc_info['location_name']}, {loc_info['city']}, "
        f"{loc_info['country']}<br>"
        f"<b>Full Address:</b> {loc_info['full_address']}<br>"
        f"<b>Target Date:</b> {target_date} "
        f"(actual image: {gee_result['image_date']})<br>"
        f"<b>Satellite Source:</b> {gee_result['source']}<br>"
        f"<b>Bands:</b> {band_str}<br>"
        f"<b>Vegetation Score:</b> {ml_result['vegetation_score']:.4f}<br>"
        f"<b>Avalanche Score:</b> {ml_result['avalanche_score']:.4f}<br>"
        f"<b>Combined Score:</b> {ml_result['combined_score']:.4f}<br>"
        f"</div>"
    )
    clear_output(wait=True)
    display(widgets.HTML(value=summary_html))
    print("\n[DONE] Data has been passed to the ML pipeline successfully.")
    set_status("All stages complete!")


run_btn.on_click(run_pipeline)